In [53]:
def format_final_output_multiple(text, entities, linker_results):

    final = {
        "intent": "product_search",
        "entities": []
    }

    product_map = {}
    id_counter = 1

    ########################################
    # CREATE PRODUCT ENTRIES
    ########################################

    for ent in entities:

        if ent["label"] == "product_name":

            product_entry = {
                "id": id_counter,
                "product_name": ent["text"],
                "is_main_product": ent.get("is_main_product", False)
            }

            product_map[ent["text"]] = product_entry

            final["entities"].append(product_entry)

            id_counter += 1

    ########################################
    # PROCESS LINKS
    ########################################

    for link in linker_results:

        attr = link["attribute"]
        product = link["product"]

        attr_label = None

        for e in entities:
            if e["text"] == attr:
                attr_label = e["label"]
                break

        ####################################
        # ATTRIBUTE → PRODUCT
        ####################################

        if attr_label not in ["product_name", "accessory"]:

            if product in product_map:
                product_map[product][attr_label] = attr

        ####################################
        # ACCESSORY → PRODUCT (SPECIAL CASE)
        ####################################

        elif attr_label == "accessory":

            parent_id = product_map[product]["id"]

            accessory_entry = {
                "id": id_counter,
                "product_name": attr,
                "is_main_product": False,
                "is_related_to": parent_id
            }

            final["entities"].append(accessory_entry)

            id_counter += 1

    #return final
    return json.dumps(final, indent=2)

In [45]:
text = "8gb samsung galaxy s23 vs 32gb chromebook"

In [47]:
entities = [
 {'label': 'memory', 'text': '8gb', 'start': 0, 'end': 3},
 {'label': 'product_name', 'text': 'samsung galaxy s23', 'start': 4, 'end': 22,"is_main_product":True},
 {'label': 'memory', 'text': '32gb', 'start': 26, 'end': 30},
 {'label': 'product_name', 'text': 'chromebook', 'start': 31, 'end': 41,"is_main_product":False}
 ]

In [49]:
linker_results = [
 {'attribute': '8gb', 'product': 'samsung galaxy s23', 'score': 0.599, 'threshold': 0.4},
{'attribute': '32gb', 'product': 'chromebook', 'score': 0.439, 'threshold': 0.4}

]

In [61]:
print(format_final_output_single(text,entities,linker_results))

{
  "intent": "product_search",
  "entities": [
    {
      "id": 1,
      "product_name": "samsung galaxy s23",
      "is_main_product": true,
      "memory": "8gb"
    },
    {
      "id": 2,
      "product_name": "chromebook",
      "is_main_product": false,
      "memory": "32gb"
    }
  ]
}


In [59]:
import json

def format_final_output_single(text, entities, linker_results=None):

    final = {
        "intent": "product_search",
        "entities": []
    }

    product_like = [
        e for e in entities
        if e["label"] in ["product_name", "accessory"]
    ]

    ########################################
    # SKIP LINKING IF ONLY ONE PRODUCT
    ########################################

    if len(product_like) <= 1:

        ent = product_like[0]

        final["entities"].append({
            "id": 1,
            "product_name": ent["text"],
            "is_main_product": ent.get("is_main_product", True)
        })

        return json.dumps(final, indent=2)

    ########################################
    # NORMAL PIPELINE
    ########################################

    product_map = {}
    id_counter = 1

    # create product entities
    for ent in entities:

        if ent["label"] == "product_name":

            product_entry = {
                "id": id_counter,
                "product_name": ent["text"],
                "is_main_product": ent.get("is_main_product", False)
            }

            product_map[ent["text"]] = product_entry

            final["entities"].append(product_entry)

            id_counter += 1

    # process linker
    if linker_results:

        for link in linker_results:

            attr = link["attribute"]
            product = link["product"]

            attr_label = None

            for e in entities:
                if e["text"] == attr:
                    attr_label = e["label"]
                    break

            ####################################
            # NORMAL ATTRIBUTES
            ####################################

            if attr_label not in ["product_name", "accessory"]:

                if product in product_map:
                    product_map[product][attr_label] = attr

            ####################################
            # ACCESSORY SPECIAL CASE
            ####################################

            elif attr_label == "accessory":

                parent_id = product_map[product]["id"]

                accessory_entry = {
                    "id": id_counter,
                    "product_name": attr,
                    "is_main_product": False,
                    "is_related_to": parent_id
                }

                final["entities"].append(accessory_entry)

                id_counter += 1

    return json.dumps(final, indent=2)

In [63]:
text = "show me s23"

In [65]:
entities = [
 {'label': 'product_name', 'text': 's23', 'start': 8, 'end': 11,"is_main_product":True},
 ]

In [67]:
linker_results=None

In [69]:
print(format_final_output_single(text,entities,linker_results))

{
  "intent": "product_search",
  "entities": [
    {
      "id": 1,
      "product_name": "s23",
      "is_main_product": true
    }
  ]
}


# Tuning Thr

In [88]:
import numpy as np

for thr in np.arange(0.1,1.1, 0.1):
    print(thr)

0.1
0.2
0.30000000000000004
0.4
0.5
0.6
0.7000000000000001
0.8
0.9
1.0
